# 6. 아파트 데이터 전처리 및 통합

**입력**
- `apt_merged_2019_2023.csv`
- `kmeans_sgg_cluster.csv`
- `aging_rate_2019_2023.csv`
- `viirs_emd_2019_2026.csv`
- `legal_to_kosis_sgg.csv`

**출력:** `apt_ml_master.csv`

---

### 진행 순서
1. 환경 설정
2. 아파트 데이터 로드
3. 아파트 클린징 (3-1~3-4)
4. 데이터 통합 (4-1~4-3)
5. 최종 확인 및 저장 → `apt_ml_master.csv`

## 1. 환경 설정

In [50]:
import pandas as pd
import numpy as np

# --- 입력 파일 ---
APT_CSV    = '아파트실거래가/apt_merged_2019_2023.csv'
KMEANS_CSV = 'kmeans_sgg_cluster.csv'
AGING_CSV  = 'aging_rate_2019_2023.csv'
VIIRS_CSV  = 'viirs_emd_2019_2026.csv'
LEGAL_CSV  = 'legal_to_kosis_sgg.csv'

# --- 출력 파일 ---
OUTPUT_CSV = 'apt_ml_master.csv'

# --- 분석 기간 (고령화·apt·VIIRS 공통) ---
YEARS = [2019, 2020, 2021, 2022, 2023]
NL_COLS = [f'nightlight_{y}' for y in YEARS]          # viirs 원본 컬럼명
NL_OUT  = [f'nl_{y}' for y in YEARS]                  # merge 후 컬럼명

def drop_col_variants(data, bases):
    drop = []
    for c in list(data.columns):
        if c in bases or any(c == f'{b}_x' or c == f'{b}_y' for b in bases):
            drop.append(c)
    if drop:
        data.drop(columns=drop, inplace=True)
    return drop

print('설정 완료')
print(f'분석 연도: {YEARS}')
print(f'출력: {OUTPUT_CSV}')

설정 완료
분석 연도: [2019, 2020, 2021, 2022, 2023]
출력: apt_ml_master.csv


## 2. 아파트 데이터 로드

In [51]:
apt = pd.read_csv(APT_CSV, encoding='utf-8-sig', low_memory=False)

print(f'shape: {apt.shape}')
print(f'\n거래 연도: {sorted(apt["dealYear"].unique())}')
print(f'고유 시군구(KOSIS): {apt["KOSIS_SGG_CODE"].nunique()}개')
print(f'\n샘플 (5행):')
apt[['aptNm', 'dealAmount', 'dealYear', 'excluUseAr', 'floor',
     'buildYear', 'KOSIS_SGG_CODE', 'umdNm']].head()

shape: (2491756, 34)

거래 연도: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
고유 시군구(KOSIS): 245개

샘플 (5행):


,aptNm,dealAmount,dealYear,excluUseAr,floor,buildYear,KOSIS_SGG_CODE,umdNm
0,광화문스페이스본(101동~105동),"119,000",2019,147.31,1,2008,11010,사직동
1,벽산블루밍평창힐스,"99,000",2019,150.94,3,2004,11010,평창동
2,동대문,"29,000",2019,28.80,3,1966,11010,창신동
3,롯데낙천대,"57,500",2019,84.21,5,2001,11010,평창동
4,삼성,"35,500",2019,59.97,11,1998,11010,평창동


## 3. 아파트데이터 전처리

- **3-1** cdealType — 취소 거래 제거
- **3-2** floor — 확인 → `floor_num` 적용
- **3-3** dealAmount — 확인 → `dealAmount_num` 적용
- **3-4** excluUseAr — 확인 → `excluUseAr_num` 적용

### 3-1 : cdealType

In [52]:
# 1번 
before = len(apt)

print('=== cdealType 분포 ===')
# 정상/취소 거래 분포 확인
print(apt['cdealType'].astype(str).str.strip().replace('', '(거래됨)').value_counts())

# 취소 거래 데이터 드롭
cancel_mask = apt['cdealType'].astype(str).str.strip() == 'O'
n_cancel = cancel_mask.sum()
print(f'\n취소 거래: {n_cancel}건 ({n_cancel / before * 100:.1f}%)')

df = apt[~cancel_mask].copy()
print(f'\n제거 후: {len(df)}건  (원본: {before}건)')

=== cdealType 분포 ===
cdealType
(거래됨)    2396831
O          94925
Name: count, dtype: int64

취소 거래: 94925건 (3.8%)

제거 후: 2396831건  (원본: 2491756건)


### 3-2 : floor

In [53]:
before = len(df)

print(f'\n=== 형태 ===')
print(f'dtype: {df["floor"].dtype}')

# 숫자로 변환 (errors='coerce' 변환 실패시 NaN 반환)
floor_num_try = pd.to_numeric(df['floor'], errors='coerce')

print('\n=== 변환 여부 ===')
print(f'전체: {len(df)}건')
print(f'숫자 변환 성공: {floor_num_try.notna().sum()}건')
print(f'변환 실패: {floor_num_try.isna().sum()}건')

# 숫자 층 분포
valid = floor_num_try.dropna()
print(f'\n=== floor 분포 ===')
print(f'지하(음수): {(valid < 0).sum()}건')
print(f'1~5층: {((valid >= 1) & (valid <= 5)).sum()}건')
print(f'6~10층: {((valid >= 6) & (valid <= 10)).sum()}건')
print(f'11~20층: {((valid >= 11) & (valid <= 20)).sum()}건')
print(f'21층 이상: {(valid >= 21).sum()}건')

# 변환 실패 샘플 확인
if floor_num_try.isna().any():
    print(f'\n=== 변환 실패 샘플 ===')
    bad = df[floor_num_try.isna()][['aptNm', 'floor', 'dealAmount', 'umdNm']].head()
    print(bad.to_string(index=False))

# 적용
df['floor_num'] = floor_num_try
df = df[df['floor_num'].notna()].copy()

print(f'\nfloor 적용 후: {len(df)}건  (제거: {before - len(df)}건)')


=== 형태 ===
dtype: object

=== 변환 여부 ===
전체: 2396831건
숫자 변환 성공: 2396830건
변환 실패: 1건

=== floor 분포 ===
지하(음수): 136건
1~5층: 817325건
6~10층: 659519건
11~20층: 785824건
21층 이상: 134026건

=== 변환 실패 샘플 ===
               aptNm floor dealAmount umdNm
양지마을(5단지)(한양515-529)           94,500   수내동

floor 적용 후: 2396830건  (제거: 1건)


### 3-3 : dealAmount (타겟변수)

In [54]:
before = len(df)

# 문자열 처리
amt_raw = df['dealAmount'].astype(str).str.strip()

print(f'\n=== 형태 ===')
print(f'dtype: {df["dealAmount"].dtype}')
print(f'쉼표 포함: {amt_raw.str.contains(",", na=False).sum():,}건')
print(f'공백: {(amt_raw == "").sum()}건')

amt_try = (
    amt_raw.str.replace(',', '')
    .replace('', np.nan)
    .astype(float)
)

print(f'\n=== 변환 여부 ===')
print(f'성공: {amt_try.notna().sum()}건')
print(f'실패: {amt_try.isna().sum()}건')
print(f'범위 (만원): {amt_try.min()} ~ {amt_try.max()}')
print(f'중간값: {amt_try.median()}')

# 적용
df['dealAmount_num'] = amt_try
df = df[df['dealAmount_num'].notna()].copy()

print(f'\ndealAmount_num 적용 후: {len(df)}건  (제거: {before - len(df)}건)')


=== 형태 ===
dtype: object
쉼표 포함: 2,396,659건
공백: 0건

=== 변환 여부 ===
성공: 2396830건
실패: 0건
범위 (만원): 400.0 ~ 1800000.0
중간값: 25500.0

dealAmount_num 적용 후: 2396830건  (제거: 0건)


### 3-4 : excluUseAr

In [55]:
before = len(df)

# 숫자로 변환 (errors='coerce' 변환 실패시 NaN 반환)
area_try = pd.to_numeric(df['excluUseAr'], errors='coerce')

print('=== 형태 ===')
print(f'dtype: {df["excluUseAr"].dtype}')

print('\n=== 변환 여부 ===')
print(f'전체: {len(df)}건')
print(f'숫자 변환 성공: {area_try.notna().sum()}건')
print(f'변환 실패: {area_try.isna().sum()}건')

# 면적 분포
valid = area_try.dropna()
print(f'\n=== excluUseAr 분포 (㎡) ===')
print(f'0 이하: {(valid <= 0).sum()}건')
print(f'~40㎡: {(valid <= 40).sum()}건')
print(f'40~60㎡: {((valid > 40) & (valid <= 60)).sum()}건')
print(f'60~85㎡: {((valid > 60) & (valid <= 85)).sum()}건')
print(f'85~100㎡: {((valid > 85) & (valid <= 100)).sum()}건')
print(f'100㎡ 초과: {(valid > 100).sum()}건')
print(f'범위: {valid.min():.2f} ~ {valid.max():.2f}')
print(f'중간값: {valid.median():.2f}')

# 변환 실패 샘플 확인
if area_try.isna().any():
    print(f'\n=== 변환 실패 샘플 ===')
    bad = df[area_try.isna()][['aptNm', 'excluUseAr', 'dealAmount', 'umdNm']].head()
    print(bad.to_string(index=False))

# 적용
df['excluUseAr_num'] = area_try
df = df[df['excluUseAr_num'].notna() & (df['excluUseAr_num'] > 0)].copy()

print(f'\nexcluUseAr_num 적용 후: {len(df)}건  (제거: {before - len(df)}건)')

=== 형태 ===
dtype: float64

=== 변환 여부 ===
전체: 2396830건
숫자 변환 성공: 2396830건
변환 실패: 0건

=== excluUseAr 분포 (㎡) ===
0 이하: 0건
~40㎡: 155439건
40~60㎡: 870565건
60~85㎡: 1078152건
85~100㎡: 37046건
100㎡ 초과: 255628건
범위: 9.26 ~ 317.36
중간값: 74.91

excluUseAr_num 적용 후: 2396830건  (제거: 0건)


## 4. 데이터 통합

- **4-1** kmeans (`cluster_name`, `nightlight_avg`)
- **4-2** aging (`고령화율`)
- **4-3** VIIRS 시군구 집계 + merge (`nl_2019`~`nl_2023`)

### 4-1 : kmeans merge

In [56]:
before = len(df)

# 혹시 다시 실행해도 문제없게 기존 컬럼 먼저 제거
drop_col_variants(df, ['SGG_NM', 'cluster_name', 'nightlight_avg', 'SGG_CODE'])

# merge 키 타입 확인
print('=== merge 키 타입 ===')
print(f'apt KOSIS_SGG_CODE: {df["KOSIS_SGG_CODE"].dtype}')

kmeans = pd.read_csv(KMEANS_CSV, encoding='utf-8-sig')
print(f'kmeans SGG_CODE: {kmeans["SGG_CODE"].dtype}')

print('\n=== kmeans 원본 ===')
print(f'shape: {kmeans.shape}')


apt_sgg = set(df['KOSIS_SGG_CODE'].unique())
km_sgg = set(kmeans['SGG_CODE'].unique())
only_apt = sorted(apt_sgg - km_sgg)
only_km = sorted(km_sgg - apt_sgg)

print('\n=== 시군구 코드 비교 ===')
print(f'apt 시군구: {len(apt_sgg)}개')
print(f'kmeans 시군구: {len(km_sgg)}개')
print(f'apt에만 있음: {len(only_apt)}개 {only_apt}')
print(f'kmeans에만 있음: {len(only_km)}개 {only_km}')

# 합치기
df = df.merge(
    kmeans[['SGG_CODE', 'cluster_name', 'nightlight_avg', 'SGG_NM']],
    left_on='KOSIS_SGG_CODE',
    right_on='SGG_CODE',
    how='left',
)
df.drop(columns=['SGG_CODE'], inplace=True)

print('\n=== merge 결과 ===')
print(f'행 수: {len(df):,}건  (변경: {len(df) - before})')

=== merge 키 타입 ===
apt KOSIS_SGG_CODE: int64
kmeans SGG_CODE: int64

=== kmeans 원본 ===
shape: (248, 5)

=== 시군구 코드 비교 ===
apt 시군구: 245개
kmeans 시군구: 248개
apt에만 있음: 0개 []
kmeans에만 있음: 3개 [np.int64(23320), np.int64(31050), np.int64(31240)]

=== merge 결과 ===
행 수: 2,396,830건  (변경: 0)


### 4-2 : aging merge

In [57]:
before = len(df)

# 혹시 다시 실행해도 문제없게 기존 컬럼 먼저 제거
drop_col_variants(df, ['고령화율', 'SGG_CODE', 'year'])

# merge 키 타입 확인
print('=== merge 키 타입 ===')
print(f'apt KOSIS_SGG_CODE: {df["KOSIS_SGG_CODE"].dtype}')
print(f'apt dealYear: {df["dealYear"].dtype}')

aging = pd.read_csv(AGING_CSV, encoding='utf-8-sig')
aging = aging[aging['year'].isin(YEARS)].copy()
print(f'aging SGG_CODE: {aging["SGG_CODE"].dtype}')
print(f'aging year: {aging["year"].dtype}')

print('\n=== aging 원본 ===')
print(f'shape: {aging.shape}')
print(f'연도: {sorted(aging["year"].unique())}')

apt_keys = set(zip(df['KOSIS_SGG_CODE'], df['dealYear']))
aging_keys = set(zip(aging['SGG_CODE'], aging['year']))
only_apt = sorted(apt_keys - aging_keys)
only_aging = sorted(aging_keys - apt_keys)

print('\n=== (시군구, 연도) 조합 비교 ===')
print(f'apt 조합: {len(apt_keys)}개')
print(f'aging 조합: {len(aging_keys)}개')
print(f'apt에만 있음: {len(only_apt)}개 {only_apt[:5]}')
print(f'aging에만 있음: {len(only_aging)}개')

# 합치기
df = df.merge(
    aging[['SGG_CODE', 'year', '고령화율']],
    left_on=['KOSIS_SGG_CODE', 'dealYear'],
    right_on=['SGG_CODE', 'year'],
    how='left',
)

# merge 키로만 썼던 컬럼 제거
df.drop(columns=['SGG_CODE', 'year'], inplace=True)

print('\n=== merge 결과 ===')
print(f'행 수: {len(df):,}건  (변경: {len(df) - before})')

=== merge 키 타입 ===
apt KOSIS_SGG_CODE: int64
apt dealYear: int64
aging SGG_CODE: int64
aging year: int64

=== aging 원본 ===
shape: (1305, 6)
연도: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

=== (시군구, 연도) 조합 비교 ===
apt 조합: 1223개
aging 조합: 1305개
apt에만 있음: 4개 [(22520, 2019), (22520, 2020), (22520, 2021), (22520, 2022)]
aging에만 있음: 86개

=== merge 결과 ===
행 수: 2,396,830건  (변경: 0)


### 4-3 : VIIRS merge

In [58]:
before = len(df)

# 혹시 다시 실행해도 문제없게 기존 컬럼 먼저 제거
drop_col_variants(df, NL_OUT)

viirs = pd.read_csv(VIIRS_CSV, encoding='utf-8-sig')
map_sgg = pd.read_csv(LEGAL_CSV, encoding='utf-8-sig')

map_sgg = map_sgg.dropna(subset=['KOSIS_SGG_CODE']).drop_duplicates('legal5')

map_sgg['legal5'] = map_sgg['legal5'].astype(int)
map_sgg['KOSIS_SGG_CODE'] = map_sgg['KOSIS_SGG_CODE'].astype(int)

print('=== VIIRS 원본 ===')
print(f'shape: {viirs.shape}')

viirs['legal5'] = viirs['EMD_CD'].astype(str).str.zfill(8).str[:5].astype(int)
viirs = viirs.merge(
    map_sgg[['legal5', 'KOSIS_SGG_CODE']],
    on='legal5',
    how='left',
)

n_unmap = viirs['KOSIS_SGG_CODE'].isna().sum()
print(f'KOSIS 매칭 실패 읍면동: {n_unmap}행')

nl_sgg = viirs.groupby('KOSIS_SGG_CODE')[NL_COLS].mean().reset_index()
nl_sgg.rename(columns=dict(zip(NL_COLS, NL_OUT)), inplace=True)

print('\n=== 시군구 집계 ===')
print(f'shape: {nl_sgg.shape}')
print(f'추가 컬럼: {NL_OUT}')

# 타입 확인
print('\n=== merge 키 타입 ===')
print(f'apt KOSIS_SGG_CODE: {df["KOSIS_SGG_CODE"].dtype}')
print(f'VIIRS KOSIS_SGG_CODE: {nl_sgg["KOSIS_SGG_CODE"].dtype}')

apt_sgg = set(df['KOSIS_SGG_CODE'].unique())
nl_sgg_set = set(nl_sgg['KOSIS_SGG_CODE'].unique())
only_apt = sorted(apt_sgg - nl_sgg_set)
only_nl = sorted(nl_sgg_set - apt_sgg)

print('\n=== 시군구 코드 비교 ===')
print(f'apt 시군구: {len(apt_sgg)}개')
print(f'VIIRS 시군구: {len(nl_sgg_set)}개')
print(f'apt에만 있음: {len(only_apt)}개 {only_apt}')
print(f'VIIRS에만 있음: {len(only_nl)}개 {only_nl}')

# 합치기
df = df.merge(nl_sgg, on='KOSIS_SGG_CODE', how='left')

print('\n=== merge 결과 ===')
print(f'행 수: {len(df):,}건  (변경: {len(df) - before})')

=== VIIRS 원본 ===
shape: (4516, 10)
KOSIS 매칭 실패 읍면동: 0행

=== 시군구 집계 ===
shape: (248, 6)
추가 컬럼: ['nl_2019', 'nl_2020', 'nl_2021', 'nl_2022', 'nl_2023']

=== merge 키 타입 ===
apt KOSIS_SGG_CODE: int64
VIIRS KOSIS_SGG_CODE: int64

=== 시군구 코드 비교 ===
apt 시군구: 245개
VIIRS 시군구: 248개
apt에만 있음: 0개 []
VIIRS에만 있음: 3개 [np.int64(23320), np.int64(31050), np.int64(31240)]

=== merge 결과 ===
행 수: 2,396,830건  (변경: 0)


## 5. 최종 확인 및 저장

In [59]:
print('=== df 최종 shape ===')
print(f'행: {len(df):,}  |  컬럼: {df.shape[1]}개')

# 컬럼 구분
prep_cols = ['floor_num', 'dealAmount_num', 'excluUseAr_num']
merge_cols = ['SGG_NM', 'cluster_name', 'nightlight_avg', '고령화율'] + NL_OUT
apt_cols = [c for c in df.columns if c not in prep_cols + merge_cols
            and not any(c.endswith(s) for s in ('_x', '_y'))]

print(f'\n원본 apt 컬럼: {len(apt_cols)}개')
print(f'전처리 추가: {prep_cols}')
print(f'merge 추가: {merge_cols}')

print('\n=== 전체 컬럼 (dtype) ===')
print(df.dtypes.to_string())

print('\n=== merge·전처리 컬럼 결측 ===')
check_cols = [c for c in prep_cols + merge_cols if c in df.columns]
missing_defs = [c for c in prep_cols + merge_cols if c not in df.columns]

=== df 최종 shape ===
행: 2,396,830  |  컬럼: 46개

원본 apt 컬럼: 34개
전처리 추가: ['floor_num', 'dealAmount_num', 'excluUseAr_num']
merge 추가: ['SGG_NM', 'cluster_name', 'nightlight_avg', '고령화율', 'nl_2019', 'nl_2020', 'nl_2021', 'nl_2022', 'nl_2023']

=== 전체 컬럼 (dtype) ===
aptDong              object
aptNm                object
aptSeq               object
bonbun                int64
bubun                 int64
buildYear             int64
buyerGbn             object
cdealDay             object
cdealType            object
dealAmount           object
dealDay               int64
dealMonth             int64
dealYear              int64
dealingGbn           object
estateAgentSggNm     object
excluUseAr          float64
floor                object
jibun                object
landCd                int64
landLeaseholdGbn     object
rgstDate             object
roadNm               object
roadNmBonbun         object
roadNmBubun          object
roadNmCd             object
roadNmSeq             int64
roadNmSggCd 

In [60]:
# df 전체 저장
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'저장 완료 → {OUTPUT_CSV}')
print(f'{len(df):,}건 × {df.shape[1]}컬럼 (전체)')
print(f'컬럼: {list(df.columns)}')

저장 완료 → apt_ml_master.csv
2,396,830건 × 46컬럼 (전체)
컬럼: ['aptDong', 'aptNm', 'aptSeq', 'bonbun', 'bubun', 'buildYear', 'buyerGbn', 'cdealDay', 'cdealType', 'dealAmount', 'dealDay', 'dealMonth', 'dealYear', 'dealingGbn', 'estateAgentSggNm', 'excluUseAr', 'floor', 'jibun', 'landCd', 'landLeaseholdGbn', 'rgstDate', 'roadNm', 'roadNmBonbun', 'roadNmBubun', 'roadNmCd', 'roadNmSeq', 'roadNmSggCd', 'roadNmbCd', 'sggCd', 'slerGbn', 'umdCd', 'umdNm', 'KOSIS_SGG_CODE', 'LEGAL_SGG_CODE', 'floor_num', 'dealAmount_num', 'excluUseAr_num', 'cluster_name', 'nightlight_avg', 'SGG_NM', '고령화율', 'nl_2019', 'nl_2020', 'nl_2021', 'nl_2022', 'nl_2023']
